# Asari and Zador (2009): STRF and LN model performance on Wehr's recordings (CRCNS ac-1)

Paper: doi:10.1152/jn.00577.2009, Figure 4 and Methods, "Neural encoding models" and "Model
performance". Dataset: CRCNS ac-1, `unversioned` (MD5 sums printed below), the folder `wehr/` of
`crcns-ac1.zip`, which holds the recordings of Machens et al. (2004) that the paper reanalysed.

This notebook tests one claim, fig4-strf-performance-low; the other claims are in
`reproduce_asari_2009.ipynb`. Every method detail and its source is in the docstring of
`strf.py`.

| Claim | Tested here |
|---|---|
| fig4-strf-performance-low | tested |

In [ ]:
import time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import ctxdep as cd
import strf

T0 = time.time()
pd.set_option("display.width", 200)
md5 = cd.checksums()
print("crcns-ac1.zip MD5:", md5["crcns-ac1.zip"])
W = cd.summary("wehr")
strf.load_stimuli()
print("stimulus waveforms:", len(strf.STIMULI), "| loaded in %.0f s" % (time.time() - T0))

## Data check

In [ ]:
rows = []
for g in W:
    for c in g["cells"]:
        nat = collections.Counter(d.split(": Trial")[0] for d in c["descriptions"] if d.startswith("natural sound"))
        rep = [v for v in nat.values() if v >= 2]
        rows.append(dict(group=g["description"][:40], cell=c["cell"], files=len(c["files"]), natural_stimuli=len(nat),
                         stimuli_2plus_trials=len(rep), trials_median=float(np.median(list(nat.values()))) if nat else 0))
CELLS = pd.DataFrame(rows)
print(CELLS.to_string())
USE = CELLS[CELLS.stimuli_2plus_trials >= 2].cell.tolist()
print(f"\n{len(CELLS)} cells in wehr/; {len(USE)} have at least two natural sound stimuli with at least two trials, "
      "which Eq. 52 needs; the paper used 20 cells and does not list them")
missing = sorted({d.split(": Trial")[0] for g in W for c in g["cells"] for d in c["descriptions"] if d.startswith("natural sound")}
                 - {k for k in strf.STIMULI} - {"natural sound fragment (" + k + ")" for k in strf.STIMULI})
print("trial descriptions without a stimulus file of that name:", missing[:5], len(missing))

In [ ]:
cache = {}
DATA = {}
t = time.time()
cellmap = {c["cell"]: c for g in W for c in g["cells"]}
for name in USE:
    for win in (10, 20):
        DATA[(name, win)] = strf.load_cell(cellmap[name], cache, win)
d0 = DATA[(USE[0], 10)]
print("loaded %d cells in %.0f s" % (len(USE), time.time() - t))
summ = pd.DataFrame([dict(cell=n, stimuli=len(DATA[(n, 10)]), trials=sum(R.shape[0] for _, R in DATA[(n, 10)]),
                          seconds=sum(R.shape[1] for _, R in DATA[(n, 10)]) * strf.DT,
                          vm_sd=np.mean([R.std() for _, R in DATA[(n, 10)]])) for n in USE])
print(summ.round(2).to_string())
S0 = d0[0][0]
print("spectrogram: %d frames x %d bands, dB range after centring %.1f to %.1f" % (S0.shape[0], S0.shape[1], S0.min(), S0.max()))

Clock check: the trial-averaged response must follow the stimulus. A model with the 180-ms window
fitted on the true alignment is compared with one fitted after shifting the response by half the
stimulus length (circularly), on the first cell.

In [ ]:
st = DATA[(USE[0], 10)]
true = strf.fit_cell(st, strf.COUNTS[0], ln=False)
shifted = [(S, np.roll(R, R.shape[1] // 2, axis=1)) for S, R in st]
sh = strf.fit_cell(shifted, strf.COUNTS[0], ln=False)
print(f"{USE[0]}: validation performance true alignment {true['lin_va']:.3f}, shifted {sh['lin_va']:.3f}")

## fig4-strf-performance-low

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the
> predictable response power, and extending the window from 180 ms to about 4 s did not improve them
> significantly, with or without a static nonlinearity; the LN model did only slightly better.

Main reading: 10-ms Hann window, contiguous folds, all nine windows. Readings at 180 and 4052 ms:
20-ms Hann window, and folds by stimulus.

In [ ]:
rows = []
t = time.time()
for name in USE:
    for k, counts in enumerate(strf.COUNTS):
        r = strf.fit_cell(DATA[(name, 10)], counts, folds="blocks")
        rows.append(dict(cell=name, reading="hann10/blocks", **r))
    for win, folds in ((20, "blocks"), (10, "stimulus")):
        for counts in (strf.COUNTS[0], strf.COUNTS[-1]):
            r = strf.fit_cell(DATA[(name, win)], counts, folds=folds)
            rows.append(dict(cell=name, reading=f"hann{win}/{folds}", **r))
PERF = pd.DataFrame(rows)
print("fits: %d in %.0f s" % (len(PERF), time.time() - t))
print("ridge value chosen at grid position (0 to 12):", dict(collections.Counter(PERF.ridge_pos)))

In [ ]:
main = PERF[PERF.reading == "hann10/blocks"]
tab = main.groupby("window_ms")[["lin_va", "lin_tr", "ln_va", "ln_tr"]].agg(["mean", "sem"])
print(tab.round(3).to_string())
print("\nmaximum over windows of the mean: STRF validation %.3f, training %.3f; LN validation %.3f, training %.3f"
      % tuple(main.groupby("window_ms")[c].mean().max() for c in ("lin_va", "lin_tr", "ln_va", "ln_tr")))
print("criterion: at most 0.22 at every window for STRF and LN, under one reading")
for rd in PERF.reading.unique():
    d = PERF[PERF.reading == rd]
    print(rd, d.groupby("window_ms")[["lin_va", "lin_tr", "ln_va", "ln_tr"]].mean().round(3).to_dict("index"))

In [ ]:
def wil(d, col):
    a = d[d.window_ms == 180].set_index("cell")[col]
    b = d[d.window_ms == 4052].set_index("cell")[col]
    diff = (b - a).dropna()
    p = stats.wilcoxon(diff, alternative="greater").pvalue
    return diff.median(), p, int((diff > 0).sum()), len(diff)


for rd in PERF.reading.unique():
    d = PERF[PERF.reading == rd]
    for col in ("lin_va", "ln_va"):
        m, p, npos, n = wil(d, col)
        print(f"{rd:16s} {col}: 4052 minus 180 ms median {m:+.4f}, improved in {npos} of {n} cells, one-sided Wilcoxon p = {p:.4f}")
for col in ("lin_va", "ln_va"):
    piv = main.pivot(index="cell", columns="window_ms", values=col)
    print(f"Friedman over nine windows ({col}): chi2 = {stats.friedmanchisquare(*[piv[c] for c in piv.columns]).statistic:.2f}, "
          f"p = {stats.friedmanchisquare(*[piv[c] for c in piv.columns]).pvalue:.4f}")
gain = (main.ln_va - main.lin_va)
print(f"LN minus STRF, validation: mean {gain.mean():+.3f}; training: mean {(main.ln_tr - main.lin_tr).mean():+.3f}")

In [ ]:
m = main.groupby("window_ms")[["lin_va", "lin_tr", "ln_va", "ln_tr"]].mean()
fig, ax = plt.subplots(figsize=(6, 4))
for c, st_ in (("lin_va", "x-"), ("lin_tr", "o-"), ("ln_va", "x--"), ("ln_tr", "o--")):
    ax.plot(m.index / 1000, m[c], st_, label=c)
ax.set_xlabel("window length (s)"); ax.set_ylabel("model performance (Eq. 53)"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
print("plotted range: %.3f to %.3f" % (m.values.min(), m.values.max()))
plt.show()

**Result: partially reproduced, resting on the claims file's test and on the cell selection.** Over
16 cells, the mean validation (lower) performance is 0.130 to 0.152 for the STRF and 0.084 to 0.121
for the LN model at every window, below 0.22; the training (upper) estimates are 0.205 to 0.249 and
0.234 to 0.289. Extending the window improves the models a little, and the one-sided Wilcoxon test
of 4052 against 180 ms finds the improvement: median +0.016 for the STRF (11 of 16 cells, p =
0.0038) and +0.023 for the LN model (14 of 16, p = 0.017), with p from 0.0038 to 0.033 under the
other readings. The LN model is worse than the STRF on validation data (mean difference -0.038)
and better on training data (+0.040). So performance is low, as the paper says, but the absence
of an improvement and the small LN gain are not reproduced.

In [ ]:
print(f"runtime {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")